In [ ]:
"""=== Part A: Local Projections (Jordà 2005) ==="""
"""Estimate the impulse response of log real GDP and log CPI to a one-unit monetary policy shock, separately at each horizon h = 0..20, with Newey–West (HAC) SEs."""
import pandas as pd
import numpy as np
import statsmodels.api as sm        # OLS with HAC (Newey–West) standard errors
from scipy.stats import norm        # normal critical values for confidence bands
import matplotlib.pyplot as plt     # IRF plots

In [ ]:
"""Read the merged quarterly dataset built in Data_Merge.ipynb."""
df = pd.read_csv("merged_data.csv", index_col = "quarter")  # Make sure to run the Data_Merge file before it to create the merged file

In [ ]:
"""Building the LP control variable vector X_t from equation (2) in the PDF:
  - FOUR lags each of log_real_gdp, log_cpi, mp_shock, ffr
  - unemployment is used in its present value form as the equation(2) writes unrate_t with no lag range, unlike the other regressors we follow it as written. 
  So we assume it takes the present value. """

control_variables = pd.DataFrame(index=df.index)

""" .shift(k) moves a series DOWN by k rows, so at time t it returns the value from t-k."""
for k in range(1,5):
    control_variables[f"log_real_gdp_l{k}"] = df["log_real_gdp"].shift(k)
    control_variables[f"log_cpi_l{k}"] = df["log_cpi"].shift(k)
    control_variables[f"mp_shock_l{k}"] = df["mp_shock"].shift(k)
    control_variables[f"ffr_l{k}"] = df["ffr"].shift(k)

control_variables["unrate"] = df["unrate"]  # No lag used 

"""The first 4 rows are NaN (no "four quarters ago" exists yet) — they drop out at estimation time. This is expected, not a bug."""
print(control_variables.head(10))

In [ ]:
"""Core of Part A: run one OLS regression per horizon h and collect the shock
coefficient. The sequence {beta(0), beta(1), ..., beta(20)} is the impulse response."""

"""Local Projection Impulse Response Function"""

def local_projection_irf(df, controls, outcome, H=20):
    """Jordà LP for one outcome over horizons 0..H.

    For each h, estimate:  y_{t+h} = a + b*mp_shock_t + g'X_t + e
    Returns a DataFrame with columns h, beta, se, nobs (one row per horizon),
    where beta(h) is the impulse response and se(h) its Newey–West standard error.
    """
    y = df[outcome]         # outcome series
    X = pd.concat([df["mp_shock"].rename("mp_shock"), controls], axis=1) # shock + control variables

    rows = []
    for h in range(H + 1):

        """LHS = outcome led by h. shift(-h) pulls future values back to row t,
        so row t holds y(t+h). (At h=0 this is just y itself.)"""
        y_lead = y.shift(-h).rename("y_lead")            

        """Stacking LHS+RHS and drop any row with a missing value. This listwise drop
        is deliberate: it carves out the valid sample for this horizon."""
        reg = pd.concat([y_lead, X], axis=1).dropna() 

        Xmat = sm.add_constant(reg.drop(columns="y_lead"))      # adding intercept alpha
        yvec = reg["y_lead"]

        """Newey–West HAC errors with bandwidth h+1. LP residuals are serially
        correlated due to overlapping horizons share shocks; HAC corrects the Standard Erorrs"""
        fit = sm.OLS(yvec, Xmat).fit(cov_type="HAC", cov_kwds={"maxlags": h + 1})

        """ Storing the shock coefficient and its Standard Erorrs"""
        rows.append({"h": h,
                     "beta": fit.params["mp_shock"],
                     "se":   fit.bse["mp_shock"],
                     "nobs": int(fit.nobs)})
    return pd.DataFrame(rows)

"""Estimate the IRF for each outcome variable, nobs stays constant at 152 across horizons
because the leads borrow post-2007 macro data — see Data_Merge note.)"""
irf_gdp = local_projection_irf(df, control_variables, "log_real_gdp", H=20)
irf_cpi = local_projection_irf(df, control_variables, "log_cpi", H= 20)

In [ ]:
"""Creating 68% and 90% confidence bands, in % units."""

def add_bands(irf):
        """Adding 68% and 90% confidence bounds (in %) to an IRF table."""

        # A two-sided X% band leaves (1-X)/2 in each tail
        # so the upper z is ppf of the cumulative level: 68% -> ppf(0.84) ~ 1.00 ; 90% -> ppf(0.95) ~ 1.645.
        z68, z90 = norm.ppf(0.84), norm.ppf(0.95) 
        out = irf.copy()
        out["beta_pct"] = out["beta"] * 100         # log point to percent
        out["se_pct"]   = out["se"]   * 100
        out["lo68"] = out["beta_pct"] - z68 * out["se_pct"]
        out["hi68"] = out["beta_pct"] + z68 * out["se_pct"]
        out["lo90"] = out["beta_pct"] - z90 * out["se_pct"]
        out["hi90"] = out["beta_pct"] + z90 * out["se_pct"]
        return out

irf_gdp_b = add_bands(irf_gdp)
irf_cpi_b = add_bands(irf_cpi)

In [ ]:
"""Plotting IRF Functions with 68% and 90% bands as per the specifications mentioned for the graphs"""
def plot_irf(irf_b, title, ylabel, fname):
    
    h = irf_b["h"]
    fig, ax = plt.subplots(figsize=(7, 4.5))

    # 90% band: lighter shading drawn first, so it sits behind
    ax.fill_between(h, irf_b["lo90"], irf_b["hi90"],
                    color="steelblue", alpha=0.20, label="90% CI")
    # 68% band: darker shading drawn on top
    ax.fill_between(h, irf_b["lo68"], irf_b["hi68"],
                    color="steelblue", alpha=0.40, label="68% CI")
   
    ax.plot(h, irf_b["beta_pct"], color="navy", lw=2, label="Point estimate")       # IRF Line
    
    ax.axhline(0, color="black", lw=0.8, ls="--")   # horizontal zero line

    ax.set_title(title)
    ax.set_xlabel("Quarters after shock")
    ax.set_ylabel(ylabel)
    ax.set_xticks(range(0, 21, 2))
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(fname, dpi=150)
    plt.show()

plot_irf(irf_gdp_b,
         title="Response of Real GDP to a Monetary Policy Shock (Local Projections)",
         ylabel="Real GDP response (% deviation)",
         fname="irf_gdp_lp.png")

plot_irf(irf_cpi_b,
         title="Response of CPI to a Monetary Policy Shock (Local Projections)",
         ylabel="CPI response (% deviation)",
        fname="irf_cpi_lp.png")